# 03 — Thử nghiệm mô hình và khóa phương án lựa chọn

Notebook so sánh RFM dựa trên quy tắc, KMeans, Gaussian Mixture, Agglomerative và DBSCAN. Hình học nội tại là điều kiện cần nhưng chưa đủ: phương án phân khúc được chọn còn phải ổn định, bao phủ khách hàng, tránh phân khúc quá nhỏ và phân tách được hành vi tương lai trong tháng 9–10.


In [1]:
from pathlib import Path
import json, os, warnings
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "online_retail_II.csv"
OUT = ROOT / "outputs"
INTERMEDIATE = OUT / "intermediate"
REPORTS = OUT / "reports"
FIGURES = OUT / "figures"
MODELS = OUT / "models"
for directory in (INTERMEDIATE, REPORTS, FIGURES, MODELS):
    directory.mkdir(parents=True, exist_ok=True)
(ROOT / "experiments").mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
DEV_CUTOFF = pd.Timestamp("2010-08-31 23:59:59")
VALIDATION_END = pd.Timestamp("2010-10-31 23:59:59")
HOLDOUT_END = pd.Timestamp("2010-12-09 23:59:59")
print(f"Thư mục gốc dự án: {ROOT}")

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
import joblib

Thư mục gốc dự án: C:\Users\lqb46\Documents\Projects\Clustomer


In [2]:
features = pd.read_csv(INTERMEDIATE / "development_features.csv", index_col="Customer ID")
outcomes = pd.read_csv(INTERMEDIATE / "validation_outcomes.csv", index_col="Customer ID")
contract = json.loads((REPORTS / "feature_contract.json").read_text(encoding="utf-8"))
feature_sets = contract["candidate_feature_sets"]
print(features.shape, outcomes.shape, feature_sets)

(3065, 9) (3065, 3) {'core_rfm': ['Recency', 'Frequency', 'Monetary'], 'behavioral': ['Recency', 'Frequency', 'Monetary', 'AOV', 'AvgBasketSize', 'ProductDiversity']}


## 1. Chỉ số đánh giá và ràng buộc kinh doanh

In [3]:
def eta_squared(values, labels):
    frame = pd.DataFrame({"y": np.log1p(values), "label": labels})
    overall = frame.y.mean(); total = ((frame.y-overall)**2).sum()
    between = sum(len(g)*(g.y.mean()-overall)**2 for _, g in frame.groupby("label"))
    return float(between/total) if total else 0.0

def metrics(X, labels, future, stability=1.0):
    labels = np.asarray(labels); assigned = labels >= 0; unique = np.unique(labels[assigned])
    shares = pd.Series(labels[assigned]).value_counts(normalize=True)
    if len(unique) < 2 or assigned.sum() < 3:
        return None
    return {
        "silhouette": silhouette_score(X[assigned], labels[assigned]),
        "davies_bouldin": davies_bouldin_score(X[assigned], labels[assigned]),
        "calinski_harabasz": calinski_harabasz_score(X[assigned], labels[assigned]),
        "coverage": assigned.mean(), "min_segment_share": shares.min(),
        "future_revenue_eta2": eta_squared(future.loc[assigned,"FutureRevenue"], labels[assigned]),
        "future_active_eta2": eta_squared(future.loc[assigned,"FutureActive"], labels[assigned]),
        "stability_ari": stability, "n_segments": len(unique),
    }

# Các ràng buộc về khả năng hành động đã được duyệt trước khi thử nghiệm.
MIN_SHARE = 0.05
MIN_COVERAGE = 0.85

## 2. Đường cơ sở kinh doanh: quy tắc tứ phân vị RFM

In [4]:
r_score = pd.qcut(features["Recency"].rank(method="first"), 4, labels=[4,3,2,1]).astype(int)
f_score = pd.qcut(features["Frequency"].rank(method="first"), 4, labels=[1,2,3,4]).astype(int)
m_score = pd.qcut(features["Monetary"].rank(method="first"), 4, labels=[1,2,3,4]).astype(int)
rule_labels = pd.cut(r_score+f_score+m_score, bins=[2,5,8,10,12], labels=[0,1,2,3], include_lowest=True).astype(int)
rule_profile = outcomes.assign(Segment=rule_labels).groupby("Segment").agg(
    Customers=("FutureRevenue","size"), FutureRevenueMean=("FutureRevenue","mean"),
    FutureActiveRate=("FutureActive","mean"))
display(rule_profile)
print("Eta bình phương doanh thu tương lai của đường cơ sở theo quy tắc:", round(eta_squared(outcomes.FutureRevenue, rule_labels), 4))

,Customers,FutureRevenueMean,FutureActiveRate
Segment,,,
0,941,100.950,0.258
1,932,223.050,0.414
2,580,357.977,0.571
3,612,"1,166.590",0.788


Eta bình phương doanh thu tương lai của đường cơ sở theo quy tắc: 0.1856


## 3. Tìm kiếm theo thuật toán và bộ đặc trưng

In [5]:
rows=[]; fitted={}
for fs_name, cols in feature_sets.items():
    scaler=StandardScaler(); X=scaler.fit_transform(np.log1p(features[cols]))
    for k in range(3,7):
        seed_labels=[]
        for seed in [11,23,42,71,101]:
            seed_labels.append(KMeans(n_clusters=k, n_init=20, random_state=seed).fit_predict(X))
        stability=np.mean([adjusted_rand_score(seed_labels[0], z) for z in seed_labels[1:]])
        model=KMeans(n_clusters=k,n_init=30,random_state=RANDOM_STATE); labels=model.fit_predict(X)
        row=metrics(X,labels,outcomes,stability); row.update(model="kmeans",feature_set=fs_name,config=f"k={k}")
        rows.append(row); fitted[("kmeans",fs_name,f"k={k}")]=(scaler,model,labels,cols)

        gmm_labels=[]
        for seed in [11,23,42]:
            gmm_labels.append(GaussianMixture(n_components=k,covariance_type="full",reg_covar=1e-5,random_state=seed).fit_predict(X))
        stability=np.mean([adjusted_rand_score(gmm_labels[0],z) for z in gmm_labels[1:]])
        model=GaussianMixture(n_components=k,covariance_type="full",reg_covar=1e-5,random_state=RANDOM_STATE); labels=model.fit_predict(X)
        row=metrics(X,labels,outcomes,stability); row.update(model="gmm",feature_set=fs_name,config=f"k={k}")
        rows.append(row); fitted[("gmm",fs_name,f"k={k}")]=(scaler,model,labels,cols)

        model=AgglomerativeClustering(n_clusters=k,linkage="ward"); labels=model.fit_predict(X)
        row=metrics(X,labels,outcomes,1.0); row.update(model="agglomerative",feature_set=fs_name,config=f"k={k}")
        rows.append(row); fitted[("agglomerative",fs_name,f"k={k}")]=(scaler,model,labels,cols)

    for eps in [0.35,0.5,0.65,0.8,1.0]:
        model=DBSCAN(eps=eps,min_samples=20); labels=model.fit_predict(X)
        row=metrics(X,labels,outcomes,1.0)
        if row:
            row.update(model="dbscan",feature_set=fs_name,config=f"eps={eps},min=20")
            rows.append(row); fitted[("dbscan",fs_name,f"eps={eps},min=20")]=(scaler,model,labels,cols)
results=pd.DataFrame(rows)
display(results.sort_values("silhouette",ascending=False).head(15))

,silhouette,davies_bouldin,calinski_harabasz,coverage,min_segment_share,future_revenue_eta2,future_active_eta2,stability_ari,n_segments,model,feature_set,config
3,0.375,0.941,"2,427.878",1.000,0.143,0.193,0.151,0.996,4,kmeans,core_rfm,k=4
0,0.353,1.033,"2,532.852",1.000,0.166,0.186,0.146,0.999,3,kmeans,core_rfm,k=3
13,0.337,1.005,"1,961.110",0.938,0.396,0.096,0.087,1.000,2,dbscan,core_rfm,"eps=0.5,min=20"
6,0.329,0.984,"2,247.775",1.000,0.077,0.212,0.162,0.978,5,kmeans,core_rfm,k=5
2,0.328,1.105,"2,210.456",1.000,0.147,0.185,0.143,1.000,3,agglomerative,core_rfm,k=3
5,0.320,1.033,"2,031.953",1.000,0.147,0.186,0.143,1.000,4,agglomerative,core_rfm,k=4
1,0.284,1.141,"1,988.298",1.000,0.230,0.138,0.115,0.999,3,gmm,core_rfm,k=3
9,0.280,1.071,"2,104.440",1.000,0.048,0.210,0.159,0.819,6,kmeans,core_rfm,k=6
8,0.264,1.149,"1,822.024",1.000,0.109,0.189,0.146,1.000,5,agglomerative,core_rfm,k=5
28,0.259,1.357,"1,073.727",0.799,0.398,0.097,0.088,1.000,2,dbscan,behavioral,"eps=0.8,min=20"


## 4. Bảng điểm có ràng buộc và lựa chọn mô hình

In [6]:
results["eligible"]=(results.min_segment_share>=MIN_SHARE)&(results.coverage>=MIN_COVERAGE)&(results.n_segments.between(3,6))
eligible=results.loc[results.eligible].copy()
def minmax(s, higher=True):
    z=(s-s.min())/(s.max()-s.min()) if s.max()>s.min() else pd.Series(.5,index=s.index)
    return z if higher else 1-z
eligible["composite"]=(.25*minmax(eligible.silhouette)+.15*minmax(eligible.davies_bouldin,False)+
    .25*minmax(eligible.future_revenue_eta2)+.10*minmax(eligible.future_active_eta2)+
    .20*minmax(eligible.stability_ari)+.05*minmax(eligible.min_segment_share))
eligible=eligible.sort_values("composite",ascending=False)
display(eligible.head(15))
assert len(eligible), "Không ứng viên nào đạt các ràng buộc về khả năng hành động"
winner=eligible.iloc[0]
key=(winner.model,winner.feature_set,winner.config)
scaler,model,labels,cols=fitted[key]
print("PHƯƠNG ÁN ĐƯỢC KHÓA:",winner[["model","feature_set","config","composite","silhouette","future_revenue_eta2","stability_ari","coverage","min_segment_share"]].to_dict())

,silhouette,davies_bouldin,calinski_harabasz,coverage,min_segment_share,future_revenue_eta2,future_active_eta2,stability_ari,n_segments,model,feature_set,config,eligible,composite
6,0.329,0.984,"2,247.775",1.000,0.077,0.212,0.162,0.978,5,kmeans,core_rfm,k=5,True,0.889
3,0.375,0.941,"2,427.878",1.000,0.143,0.193,0.151,0.996,4,kmeans,core_rfm,k=4,True,0.877
0,0.353,1.033,"2,532.852",1.000,0.166,0.186,0.146,0.999,3,kmeans,core_rfm,k=3,True,0.825
5,0.320,1.033,"2,031.953",1.000,0.147,0.186,0.143,1.000,4,agglomerative,core_rfm,k=4,True,0.788
2,0.328,1.105,"2,210.456",1.000,0.147,0.185,0.143,1.000,3,agglomerative,core_rfm,k=3,True,0.781
8,0.264,1.149,"1,822.024",1.000,0.109,0.189,0.146,1.000,5,agglomerative,core_rfm,k=5,True,0.737
20,0.224,1.285,"1,183.893",1.000,0.129,0.192,0.146,0.986,5,kmeans,behavioral,k=5,True,0.692
17,0.230,1.363,"1,260.355",1.000,0.200,0.181,0.135,0.989,4,kmeans,behavioral,k=4,True,0.648
14,0.252,1.245,"1,494.148",1.000,0.261,0.164,0.125,0.995,3,kmeans,behavioral,k=3,True,0.618
22,0.181,1.486,948.725,1.000,0.064,0.177,0.132,1.000,5,agglomerative,behavioral,k=5,True,0.559


PHƯƠNG ÁN ĐƯỢC KHÓA: {'model': 'kmeans', 'feature_set': 'core_rfm', 'config': 'k=5', 'composite': 0.8885708900375917, 'silhouette': 0.32856830916877494, 'future_revenue_eta2': 0.21225595178504097, 'stability_ari': 0.9783097244384695, 'coverage': 1.0, 'min_segment_share': 0.0766721044045677}


In [7]:
profile=features.assign(Cluster=labels).groupby("Cluster").agg(
    Customers=("Frequency","size"),RecencyMedian=("Recency","median"),
    FrequencyMedian=("Frequency","median"),MonetaryMedian=("Monetary","median"),
    MonetaryMean=("Monetary","mean"))
profile["CustomerShare"]=profile.Customers/len(features)
future_profile=outcomes.assign(Cluster=labels).groupby("Cluster").agg(
    FutureRevenueMean=("FutureRevenue","mean"),FutureActiveRate=("FutureActive","mean"))
display(profile.join(future_profile))

,Customers,RecencyMedian,FrequencyMedian,MonetaryMedian,MonetaryMean,CustomerShare,FutureRevenueMean,FutureActiveRate
Cluster,,,,,,,,
0,760,97.000,2.000,735.920,939.135,0.248,282.008,0.480
1,235,8.000,12.000,"4,806.260","9,257.895",0.077,"2,144.981",0.889
2,568,34.000,5.000,"1,779.750","2,117.349",0.185,551.057,0.708
3,1087,153.000,1.000,208.000,227.694,0.355,94.544,0.255
4,415,16.000,2.000,468.890,559.226,0.135,217.564,0.455


## 5. Theo dõi bảng thử nghiệm và khóa artifact

In [8]:
try:
    import logging
    import mlflow
    logging.getLogger("mlflow").setLevel(logging.WARNING)
    logging.getLogger("alembic").setLevel(logging.WARNING)
    mlflow.set_tracking_uri(f"sqlite:///{(ROOT/'experiments'/'mlflow.db').resolve().as_posix()}")
    mlflow.set_experiment("clustomer-segmentation")
    for _, row in results.iterrows():
        with mlflow.start_run(run_name=f"{row.model}-{row.feature_set}-{row.config}"):
            mlflow.log_params({"model":row.model,"feature_set":row.feature_set,"config":row.config})
            mlflow.log_metrics({k:float(row[k]) for k in ["silhouette","davies_bouldin","coverage","min_segment_share","future_revenue_eta2","future_active_eta2","stability_ari"]})
except Exception as exc:
    print("Cảnh báo khi ghi log MLflow:",exc)
selection={"model":winner.model,"feature_set":winner.feature_set,"config":winner.config,
           "features":list(cols),"random_state":RANDOM_STATE,
           "development_metrics":{k:float(winner[k]) for k in ["composite","silhouette","davies_bouldin","calinski_harabasz","coverage","min_segment_share","future_revenue_eta2","future_active_eta2","stability_ari"]}}
results.to_csv(REPORTS/"experiment_scorecard.csv",index=False)
(REPORTS/"locked_selection.json").write_text(json.dumps(selection,indent=2),encoding="utf-8")
joblib.dump({"scaler":scaler,"model":model,"features":list(cols),"labels":labels},MODELS/"development_model.joblib")
features.assign(Cluster=labels).to_csv(INTERMEDIATE/"development_segments.csv")
print(json.dumps(selection,indent=2))

{
  "model": "kmeans",
  "feature_set": "core_rfm",
  "config": "k=5",
  "features": [
    "Recency",
    "Frequency",
    "Monetary"
  ],
  "random_state": 42,
  "development_metrics": {
    "composite": 0.8885708900375917,
    "silhouette": 0.32856830916877494,
    "davies_bouldin": 0.9843047915964325,
    "calinski_harabasz": 2247.774893318967,
    "coverage": 1.0,
    "min_segment_share": 0.0766721044045677,
    "future_revenue_eta2": 0.21225595178504097,
    "future_active_eta2": 0.16229793469198606,
    "stability_ari": 0.9783097244384695
  }
}


## Quyết định lựa chọn mô hình

Phương án đứng đầu chỉ được chọn trong số các ứng viên đạt ràng buộc về độ bao phủ, quy mô và số lượng phân khúc. Điểm tổng hợp cân bằng hình học cụm, tính hữu ích trong giai đoạn xác thực và độ ổn định giữa các seed. DBSCAN không đủ điều kiện nếu khả năng phân tách biểu kiến phụ thuộc vào việc gắn quá nhiều khách hàng là nhiễu. Thiết kế hiện đã được khóa; tập kiểm định cuối tháng 11–12 vẫn chưa được mở.
